# Open Targets — API & bulk

Open Targets ships two surfaces in `bioDB`:

| Mode | Module | When to reach for it |
|---|---|---|
| **API (GraphQL)** | `biodb.opentargets_graphql` | One target / disease / drug at a time |
| **Bulk (Parquet FTP)** | `biodb.opentargets` | Whole-corpus analysis — gene-association matrices, expression, essentiality |

## 1. API mode — GraphQL targeted lookups

Lightweight `httpx`-based client with exponential backoff. No extras needed.

In [1]:
from biodb import opentargets_graphql as otg

brca1 = otg.query_target("ENSG00000012048")
{k: brca1[k] for k in ("id", "approvedSymbol", "approvedName", "biotype")}

{'id': 'ENSG00000012048',
 'approvedSymbol': 'BRCA1',
 'approvedName': 'BRCA1 DNA repair associated',
 'biotype': 'protein_coding'}

In [2]:
breast_ca = otg.query_disease("EFO_0000305")
{k: breast_ca[k] for k in ("id", "name", "description")}

{'id': 'EFO_0000305',
 'name': 'breast carcinoma',
 'description': 'A carcinoma that arises from epithelial cells of the breast'}

In [3]:
aspirin = otg.query_drug("CHEMBL25")
{k: aspirin[k] for k in ("id", "name", "drugType", "maximumClinicalStage")}

{'id': 'CHEMBL25',
 'name': 'ASPIRIN',
 'drugType': 'Small molecule',
 'maximumClinicalStage': 'APPROVAL'}

## 2. Bulk mode — Parquet FTP

`list_datasets()` enumerates every Open Targets release dataset for the
pinned version (`opentargets.DEFAULT_VERSION`, currently `25.12`).

In [4]:
from biodb import opentargets

datasets = opentargets.list_datasets()
print(f"{len(datasets)} datasets available in OT {opentargets.DEFAULT_VERSION}")
list(datasets)[:8]

56 datasets available in OT 25.12


['association_by_datasource_direct',
 'association_by_datasource_indirect',
 'association_by_datatype_direct',
 'association_by_datatype_indirect',
 'association_overall_direct',
 'association_overall_indirect',
 'biosample',
 'colocalisation']

`get_dataset(name)` fetches the parquet shards for one dataset and
returns a pandas DataFrame by default (`output_format="polars"` for
polars). The built-in `limit=` parameter trims rows before parsing —
convenient when you only need a peek. Shards are cached at
`~/.cache/biodb/opentargets/<version>/<dataset>/`.

In [5]:
disease = opentargets.get_dataset("disease", limit=5, verbose=0)
disease[["id", "name", "description"]]

,id,name,description
0,DOID_0050890,synucleinopathy,A neurodegenerative disease that is characteri...
1,DOID_10113,trypanosomiasis,Infection with protozoa of the genus trypanosoma.
2,DOID_10718,giardiasis,An infection of the small intestine caused by ...
3,DOID_13406,pulmonary sarcoidosis,Sarcoidosis affecting the lung parenchyma. It ...
4,DOID_1947,trichomoniasis,An infection that is caused by Trichomonas.
